# 06 — Final analysis: TEST, decision layer, baseline, performance, robustness

**Runtime:** CPU, ~2 min, from committed result files (notebook 04 outputs). No dataset needed.

| § | Question | Work package |
|---|---|---|
| 1 | How good is the selected model on data never used for any choice? | WP7 |
| 2 | Do the pre-registered PASS/REVIEW/FAIL thresholds work, and when is the system worth using? | WP5 |
| 3 | Do we need a deep detector? (scikit-learn alternative) | WP4 |
| 4 | Is it fast enough, and *why* does it run at that speed? (latency, FLOPs, roofline, profiler) | WP6 |
| 5 | What do input resolution and FP16 trade off? | WP6 |
| 6 | How robust is it to lighting, blur, noise and compression? | WP7 |

Headline metric where noted: mean AP50 over **unsealed + packaging-error**, the two classes with
enough instances to evaluate (D-024); all-class mAP50 is reported alongside but is dominated by
classes with 1–4 instances. Confidence intervals resample **50-frame runs** (V12).

In [1]:
import sys, json, subprocess
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, yaml
from porkqc import CLASS_NAMES
from porkqc.plotting import apply_style, SERIES, MUTED, INK
from porkqc.predictions import load_json, pred_arrays
from porkqc.data.audit import frame_number
from porkqc.metrics.evaluate import match_dataset, summarise, bootstrap, paired_bootstrap
from porkqc.decision import image_scores, decide, evaluate_decisions, DecisionCosts, PASS, REVIEW, FAIL
apply_style()
R = ROOT / "results"; FIG = R / "figures"; OUT = R / "final"; OUT.mkdir(exist_ok=True)
UNS, PKG = CLASS_NAMES.index("unsealed"), CLASS_NAMES.index("packaging-error")

def matches(gt, pred_path):
    m = match_dataset(gt, load_json(pred_path), len(CLASS_NAMES))
    for x in m:
        x.group = str(frame_number(x.file) // 50)
    return m
def headline(mm):
    pc = summarise(mm, CLASS_NAMES)["per_class"]
    vals = [pc[c]["ap50"] for c in (UNS, PKG)]
    return float(np.mean(vals)) if None not in vals else float("nan")
def ci(mm, f, n=1000):
    b = bootstrap(mm, f, n_boot=n, seed=0, grouped=True)
    return b["point"], b["low"], b["high"]

gt_valid = load_json(R / "ground_truth/blocked_valid.json")
gt_test = load_json(R / "test/blocked_test_gt.json")
m_valid = matches(gt_valid, R / "perf/valid_imgsz640_fp32.json")
m_test = matches(gt_test, R / "test/yolo11n_640_blocked_blocked_test.json")
print(f"VALID {len(m_valid)} images / {len({x.group for x in m_valid})} runs; TEST {len(m_test)} images / {len({x.group for x in m_test})} runs")

VALID 196 images / 18 runs; TEST 180 images / 30 runs


## 1. One-shot TEST result (selected model: YOLO11n, blocked split)

In [2]:
rows = []
for name, mm in (("VALID (used for all choices)", m_valid), ("TEST (used once)", m_test)):
    s = summarise(mm, CLASS_NAMES); pc = {c["class"]: c for c in s["per_class"]}
    for metric, f in (("mAP50 (all classes with GT)", lambda x: summarise(x, CLASS_NAMES)["map50"]),
                      ("mAP50-95", lambda x: summarise(x, CLASS_NAMES)["map50_95"]),
                      ("headline AP50 (unsealed+packaging)", headline),
                      ("AP50 unsealed", lambda x: summarise(x, CLASS_NAMES)["per_class"][UNS]["ap50"]),
                      ("AP50 packaging-error", lambda x: summarise(x, CLASS_NAMES)["per_class"][PKG]["ap50"])):
        p, lo, hi = ci(mm, f)
        rows.append({"set": name, "metric": metric, "value": p, "95% CI low": lo, "95% CI high": hi})
test_table = pd.DataFrame(rows).pivot(index="metric", columns="set").round(3)
display(test_table)
inst = pd.DataFrame({n: {c["class"]: c["n_gt"] for c in summarise(mm, CLASS_NAMES)["per_class"]} for n, mm in
                     (("VALID boxes", m_valid), ("TEST boxes", m_test))})
display(inst.T)
pd.DataFrame(rows).to_csv(OUT / "test_vs_valid.csv", index=False)

tp = load_json(R / "test/yolo11n_640_blocked_blocked_test.json")
env = load_json(R / "notebook04_env.json")
print("Decision thresholds frozen at commit:", tp["decision_config_commit"])
print("TEST predictions generated at (UTC):", env["timestamp_utc"], "| code commit", tp["git_commit"])

value  \
set                                TEST (used once)   
metric                                                
AP50 packaging-error                          0.585   
AP50 unsealed                                 0.912   
headline AP50 (unsealed+packaging)            0.748   
mAP50 (all classes with GT)                   0.753   
mAP50-95                                      0.480   

                                                                 \
set                                VALID (used for all choices)   
metric                                                            
AP50 packaging-error                                      0.687   
AP50 unsealed                                             0.807   
headline AP50 (unsealed+packaging)                        0.747   
mAP50 (all classes with GT)                               0.609   
mAP50-95                                                  0.335   

                                         95% CI low  \
set                                TEST (used once)   
metric                                                
AP50 packaging-error                          0.477   
AP50 unsealed                                 0.830   
headline AP50 (unsealed+packaging)            0.680   
mAP50 (all classes with GT)                   0.514   
mAP50-95                                      0.285   

                                                                 \
set                                VALID (used for all choices)   
metric                                                            
AP50 packaging-error                                      0.637   
AP50 unsealed                                             0.674   
headline AP50 (unsealed+packaging)                        0.673   
mAP50 (all classes with GT)                               0.501   
mAP50-95                                                  0.255   

                                        95% CI high  \
set                                TEST (used once)   
metric                                                
AP50 packaging-error                          0.725   
AP50 unsealed                                 0.991   
headline AP50 (unsealed+packaging)            0.838   
mAP50 (all classes with GT)                   0.894   
mAP50-95                                      0.566   

                                                                 
set                                VALID (used for all choices)  
metric                                                           
AP50 packaging-error                                      0.745  
AP50 unsealed                                             0.912  
headline AP50 (unsealed+packaging)                        0.818  
mAP50 (all classes with GT)                               0.803  
mAP50-95                                                  0.398

,loose-meat,packaging-error,twisted-meat,unsealed,wrinkle
VALID boxes,3,117,4,59,0
TEST boxes,1,126,2,47,0


Decision thresholds frozen at commit: 0cbee97 2026-10-17T12:28:10+11:00
TEST predictions generated at (UTC): 2026-10-17T01:42:48+00:00 | code commit 3afd7bc


**Reading.** TEST scores are *higher* than VALID, but their intervals overlap widely: TEST and
VALID are different stretches of video (30 and 18 independent tray runs), so either can land on
easier footage. What matters is that TEST was never used for a choice — the thresholds were
committed 14 minutes before the TEST predictions existed — and that, unlike the Roboflow split,
neither set contains near-twins of training frames.

## 2. Decision layer on TEST (thresholds frozen on VALID, D-026)

In [3]:
cfg = yaml.safe_load(open(ROOT / "configs/decision.yaml"))
costs = DecisionCosts(**cfg["costs"])
res = {}
for name, gt, path in (("VALID", gt_valid, R / "perf/valid_imgsz640_fp32.json"),
                       ("TEST", gt_test, R / "test/yolo11n_640_blocked_blocked_test.json")):
    s, y, files = image_scores(gt, load_json(path))
    d = decide(s, cfg["t_review"], cfg["t_fail"])
    res[name] = {"model (frozen thresholds)": evaluate_decisions(d, y, costs),
                 "always FAIL": evaluate_decisions(np.full(len(y), FAIL, object), y, costs),
                 "always PASS": evaluate_decisions(np.full(len(y), PASS, object), y, costs)}
    if name == "TEST":
        s_test, y_test, f_test, d_test = s, y, files, d
dec = pd.concat({k: pd.DataFrame(v).T for k, v in res.items()})
cols = ["n", "n_defective", "misses", "false_rejects", "reviews", "miss_rate", "false_reject_rate", "review_rate", "auto_fail_precision", "cost_per_image"]
display(dec[cols].astype(float).round(3))
dec[cols].to_csv(OUT / "decision_valid_test.csv")
print("TEST missed:", [f for f, dd, yy in zip(f_test, d_test, y_test) if dd == PASS and yy])

n  n_defective  misses  false_rejects  \
VALID model (frozen thresholds)  196.0        159.0     0.0           18.0   
      always FAIL                196.0        159.0     0.0           37.0   
      always PASS                196.0        159.0   159.0            0.0   
TEST  model (frozen thresholds)  180.0        156.0     1.0            9.0   
      always FAIL                180.0        156.0     0.0           24.0   
      always PASS                180.0        156.0   156.0            0.0   

                                 reviews  miss_rate  false_reject_rate  \
VALID model (frozen thresholds)     27.0      0.000              0.486   
      always FAIL                    0.0      0.000              1.000   
      always PASS                    0.0      1.000              0.000   
TEST  model (frozen thresholds)     24.0      0.006              0.375   
      always FAIL                    0.0      0.000              1.000   
      always PASS                    0.0      1.000              0.000   

                                 review_rate  auto_fail_precision  \
VALID model (frozen thresholds)        0.138                0.890   
      always FAIL                      0.000                0.811   
      always PASS                      0.000                  NaN   
TEST  model (frozen thresholds)        0.133                0.941   
      always FAIL                      0.000                0.867   
      always PASS                      0.000                  NaN   

                                 cost_per_image  
VALID model (frozen thresholds)           0.126  
      always FAIL                         0.189  
      always PASS                         8.112  
TEST  model (frozen thresholds)           0.139  
      always FAIL                         0.133  
      always PASS                         8.667

TEST missed: ['frame_with_red_6898_jpg.rf.02dcd3f8f22441577154093e19ebf02c.jpg']


### 2.1 When is the system worth using? Sensitivity to defect prevalence

The data are ~85% defective images, whereas a real line produces mostly good packs. With miss
rate $m$, false-reject rate $f$ and review rates $q_d, q_c$ (defective / clean images), the expected
cost per tray at defect prevalence $\pi$ is

$$C(\pi) = \pi\,(10\,m + 0.25\,q_d) + (1-\pi)\,(1\cdot f + 0.25\,q_c),$$

versus $1-\pi$ for "always FAIL" and $10\pi$ for "always PASS". Rates are taken from TEST. Caveat:
the TEST "clean" images are mostly edge frames (R1), so $f$ is estimated on the wrong population —
the curve shows the *structure* of the trade-off, not a production prediction.

In [4]:
yb = np.asarray(y_test)
m_rate = np.mean(d_test[yb] == PASS); qd = np.mean(d_test[yb] == REVIEW)
f_rate = np.mean(d_test[~yb] == FAIL); qc = np.mean(d_test[~yb] == REVIEW)
pi = np.linspace(0.001, 0.95, 400)
c_model = pi * (costs.c_miss * m_rate + costs.c_review * qd) + (1 - pi) * (costs.c_false_reject * f_rate + costs.c_review * qc)
c_fail, c_pass = (1 - pi) * costs.c_false_reject, pi * costs.c_miss
fig, ax = plt.subplots(figsize=(6.4, 3.6))
ax.plot(pi, c_model, color=SERIES[0], label="model, frozen thresholds")
ax.plot(pi, c_fail, color=SERIES[1], label="always FAIL")
ax.plot(pi, c_pass, color=SERIES[2], label="always PASS")
ax.axvline(np.mean(yb), color=MUTED, ls=":", lw=1); ax.text(np.mean(yb) - 0.01, 0.95, "TEST prevalence", ha="right", fontsize=8, color=MUTED)
ax.set_xlabel("defect prevalence on the line"); ax.set_ylabel("expected cost per tray"); ax.set_ylim(0, 1.05); ax.set_xlim(0, 0.95)
ax.legend(loc="upper center"); ax.set_title("Expected cost vs defect prevalence (miss 10 : false reject 1 : review 0.25)")
fig.tight_layout(); fig.savefig(FIG / "06_cost_vs_prevalence.png", dpi=150); plt.show()
cross = pi[np.argmax(c_model > c_fail)] if np.any(c_model > c_fail) else None
print(f"rates on TEST: miss {m_rate:.3f}, false reject {f_rate:.3f}, review defective {qd:.3f}, review clean {qc:.3f}")
print(f"model cheaper than always-FAIL for prevalence below {cross:.2f}" if cross else "model always cheaper")
for p_ in (0.01, 0.05, 0.2):
    i = np.argmin(abs(pi - p_)); print(f"prevalence {p_:.0%}: model {c_model[i]:.3f}, always FAIL {c_fail[i]:.3f}, always PASS {c_pass[i]:.3f}")

rates on TEST: miss 0.006, false reject 0.375, review defective 0.077, review clean 0.500
model cheaper than always-FAIL for prevalence below 0.86
prevalence 1%: model 0.496, always FAIL 0.989, always PASS 0.105
prevalence 5%: model 0.479, always FAIL 0.949, always PASS 0.509
prevalence 20%: model 0.416, always FAIL 0.799, always PASS 2.008


## 3. Alternative: scikit-learn image-level classifiers (do we need a deep detector?)

Features: 324 hand-crafted colour/layout/edge features (`porkqc.models.baseline`). Pipelines
`StandardScaler → {decision tree, RBF SVM, MLP}`; hyperparameters chosen by image-level AP on
VALID, then scored once on TEST. The detector is scored on the **same image-level task** (score =
its highest box confidence for that class), with the same exact-AP rule.

In [5]:
import time
from porkqc.models.baseline import select_on_validation, decision_scores, image_level_ap, image_features
F = {s: np.load(R / f"baseline/features_{s}.npz") for s in ("train", "valid", "test")}
assert list(F["test"]["files"]) == [e["file"] for e in gt_test["images"]], "feature/GT order mismatch"
det_test = load_json(R / "test/yolo11n_640_blocked_blocked_test.json"); det_by = {e["file"]: e for e in det_test["images"]}
def det_score(cls):
    out = []
    for f in F["test"]["files"]:
        c, s, _ = pred_arrays(det_by[str(f)])
        keep = np.ones_like(c, bool) if cls is None else c == cls
        out.append(s[keep].max() if keep.any() else 0.0)
    return np.array(out)
groups = np.array([frame_number(str(f)) // 50 for f in F["test"]["files"]])
def grouped_ci(y, sc, n=1000, seed=0):
    rng = np.random.default_rng(seed); g = np.unique(groups); vals = []
    for _ in range(n):
        pick = np.concatenate([np.where(groups == k)[0] for k in rng.choice(g, len(g))])
        vals.append(image_level_ap(y[pick], sc[pick]))
    v = np.array(vals); v = v[~np.isnan(v)]
    return np.quantile(v, 0.025), np.quantile(v, 0.975)

rows, fitted = [], {}
for target, cls in (("any defect", None), ("unsealed", UNS), ("packaging-error", PKG)):
    ytr = F["train"]["Y"].any(1) if cls is None else F["train"]["Y"][:, cls]
    yva = F["valid"]["Y"].any(1) if cls is None else F["valid"]["Y"][:, cls]
    yte = F["test"]["Y"].any(1) if cls is None else F["test"]["Y"][:, cls]
    for r in select_on_validation(F["train"]["X"], ytr.astype(int), F["valid"]["X"], yva.astype(int)):
        sc = decision_scores(r["pipeline"], F["test"]["X"]); lo, hi = grouped_ci(yte, sc)
        rows.append({"target": target, "model": r["family"], "chosen on VALID": str(r["params"]), "VALID AP": r["valid_ap"],
                     "TEST AP": image_level_ap(yte, sc), "TEST CI low": lo, "TEST CI high": hi, "fit s": r["fit_seconds"]})
        fitted[(target, r["family"])] = r["pipeline"]
    sc = det_score(cls); lo, hi = grouped_ci(yte, sc)
    rows.append({"target": target, "model": "YOLO11n detector", "chosen on VALID": "D-025", "VALID AP": np.nan,
                 "TEST AP": image_level_ap(yte, sc), "TEST CI low": lo, "TEST CI high": hi, "fit s": 22.6 * 60})
    rows.append({"target": target, "model": "prevalence (no skill)", "chosen on VALID": "-", "VALID AP": np.nan,
                 "TEST AP": float(np.mean(yte)), "TEST CI low": np.nan, "TEST CI high": np.nan, "fit s": 0})
base = pd.DataFrame(rows)
display(base.round(3))
base.to_csv(OUT / "baseline_vs_detector.csv", index=False)

,target,model,chosen on VALID,VALID AP,TEST AP,TEST CI low,TEST CI high,fit s
0,any defect,decision tree,"{'max_depth': 8, 'min_samples_leaf': 5}",0.873,0.903,0.848,0.957,0.214
1,any defect,SVM (RBF),"{'C': 0.3, 'gamma': 0.01}",0.887,0.941,0.907,0.974,0.591
2,any defect,MLP,"{'hidden_layer_sizes': (64,), 'alpha': 1.0}",0.877,0.959,0.931,0.981,1.132
3,any defect,YOLO11n detector,D-025,NaN,0.980,0.957,0.994,1356.000
4,any defect,prevalence (no skill),-,NaN,0.867,NaN,NaN,0.000
5,unsealed,decision tree,"{'max_depth': None, 'min_samples_leaf': 5}",0.819,0.807,0.690,0.887,0.319
6,unsealed,SVM (RBF),"{'C': 1, 'gamma': 0.01}",0.907,0.937,0.882,0.977,0.343
7,unsealed,MLP,"{'hidden_layer_sizes': (128, 64), 'alpha': 0.0...",0.894,0.953,0.913,0.984,1.931
8,unsealed,YOLO11n detector,D-025,NaN,0.973,0.930,0.999,1356.000
9,unsealed,prevalence (no skill),-,NaN,0.261,NaN,NaN,0.000


In [6]:
# Inference cost of the baseline on this CPU, using real 720x540 images from the development sample
from PIL import Image
sample = sorted((ROOT.parent / "dev_sample_images").glob("*.jpg"))[:40] if (ROOT.parent / "dev_sample_images").exists() else []
if not sample:   # fall back: synthetic 720x540 images (same size, same cost)
    rng = np.random.default_rng(0); imgs = [rng.integers(0, 255, (540, 720, 3), dtype=np.uint8) for _ in range(40)]
else:
    imgs = [np.asarray(Image.open(p).convert("RGB")) for p in sample]
t0 = time.perf_counter(); Xs = np.stack([image_features(im) for im in imgs]); t_feat = (time.perf_counter() - t0) / len(imgs) * 1e3
pipe = fitted[("any defect", "SVM (RBF)")]
t0 = time.perf_counter(); [decision_scores(pipe, Xs[i:i + 1]) for i in range(len(Xs))]; t_pred = (time.perf_counter() - t0) / len(Xs) * 1e3
lat = pd.read_csv(R / "perf/latency.csv")
cpu640 = lat[(lat.device == "cpu") & (lat.imgsz == 640)].ms_median.iloc[0]
print(f"baseline on this CPU: features {t_feat:.1f} ms + SVM {t_pred:.2f} ms per image; detector forward pass on Colab CPU {cpu640:.0f} ms, T4 GPU 7.1 ms")

baseline on this CPU: features 27.3 ms + SVM 1.07 ms per image; detector forward pass on Colab CPU 82 ms, T4 GPU 7.1 ms


## 4. Speed: is it fast enough, and why does it run at that speed?

In [7]:
lat = pd.read_csv(R / "perf/latency.csv"); fl = pd.read_csv(R / "perf/flops_intensity.csv")
display(lat[["device", "batch", "imgsz", "precision", "ms_median", "ms_q25", "ms_q75", "ms_p95", "ms_per_image", "images_per_s", "peak_mem_mb"]].round(2))
T4 = {"fp32": 8.1e12, "fp16": 65e12}; BW = 320e9
g = lat[lat.device == "cuda"].merge(fl[["imgsz", "batch", "precision", "gflops", "intensity"]], on=["imgsz", "batch", "precision"], how="left")
g["achieved_tflops"] = g.gflops / (g.ms_median / 1e3) / 1e3
g["share_of_peak"] = g.apply(lambda r: r.achieved_tflops * 1e12 / T4[r.precision], axis=1)
display(g.dropna(subset=["gflops"])[["batch", "imgsz", "precision", "gflops", "intensity", "ms_median", "achieved_tflops", "share_of_peak"]].round(3))
g.to_csv(OUT / "achieved_flops.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for k, prec in enumerate(("fp32", "fp16")):
    s = lat[(lat.device == "cuda") & (lat.imgsz == 640) & (lat.precision == prec)].drop_duplicates("batch")
    axes[0].plot(s.batch, s.images_per_s, marker="o", color=SERIES[k], label=prec.upper())
axes[0].axhline(2, color=MUTED, ls="--", lw=1); axes[0].text(1, 8, "line requirement: 2 trays/s", fontsize=8, color=MUTED)
axes[0].set_xscale("log", base=2); axes[0].set_xlabel("batch size"); axes[0].set_ylabel("images per second"); axes[0].set_title("T4 throughput (640 input)"); axes[0].legend()
s = lat[(lat.device == "cuda") & (lat.batch == 1) & (lat.precision == "fp32")].drop_duplicates("imgsz", keep="last").sort_values("imgsz")
f1 = fl[(fl.batch == 1) & (fl.precision == "fp32")].set_index("imgsz").gflops
axes[1].plot(s.imgsz, s.ms_median, marker="o", color=SERIES[0], label="measured batch-1 latency")
axes[1].plot(s.imgsz, f1.loc[s.imgsz].values / f1.loc[640] * s.set_index("imgsz").ms_median.loc[640], color=SERIES[1], ls="--",
             label="if latency scaled with FLOPs")
axes[1].set_xlabel("input size (px)"); axes[1].set_ylabel("ms per image"); axes[1].set_title("Batch-1 latency vs input size (T4, FP32)"); axes[1].legend()
I = np.logspace(0, 3, 200)
for prec, col in (("fp32", SERIES[0]), ("fp16", SERIES[1])):
    axes[2].loglog(I, np.minimum(T4[prec], I * BW) / 1e12, color=col, lw=1, label=f"T4 roof {prec.upper()}")
    pts = g[(g.precision == prec) & g.gflops.notna()]
    axes[2].scatter(pts.intensity, pts.achieved_tflops, color=col, s=36, zorder=3)
    for _, r in pts.iterrows():
        lab = f"b{r.batch}" if r.imgsz == 640 else f"{r.imgsz}px"
        axes[2].annotate(lab, (r.intensity, r.achieved_tflops), fontsize=7, xytext=(4, -3 if r.batch == 8 else 3), textcoords="offset points")
axes[2].set_xlabel("arithmetic intensity (FLOP/byte, upper bound)"); axes[2].set_ylabel("TFLOP/s"); axes[2].set_title("Roofline: achieved vs attainable"); axes[2].legend(fontsize=7)
fig.tight_layout(); fig.savefig(FIG / "06_performance.png", dpi=150); plt.show()
print((R / "perf/profiler_top_ops.txt").read_text()[:2500])

,device,batch,imgsz,precision,ms_median,ms_q25,ms_q75,ms_p95,ms_per_image,images_per_s,peak_mem_mb
0,cuda,1,640,fp32,7.06,6.93,7.33,9.02,7.06,141.73,62.82
1,cuda,1,640,fp16,8.52,8.43,8.80,11.30,8.52,117.33,48.05
2,cuda,4,640,fp32,13.91,13.74,13.95,14.15,3.48,287.59,122.97
3,cuda,4,640,fp16,8.51,8.43,8.71,11.99,2.13,469.86,91.25
4,cuda,8,640,fp32,24.91,24.70,24.94,25.01,3.11,321.18,236.03
5,cuda,8,640,fp16,12.92,12.78,12.94,15.34,1.61,619.35,136.21
6,cuda,16,640,fp32,46.31,46.18,46.50,46.61,2.89,345.51,361.23
7,cuda,16,640,fp16,25.40,25.36,25.49,25.75,1.59,629.82,218.73
8,cuda,32,640,fp32,93.88,93.40,94.13,94.57,2.93,340.85,656.41
9,cuda,32,640,fp16,48.84,48.65,49.03,49.14,1.53,655.15,365.59


,batch,imgsz,precision,gflops,intensity,ms_median,achieved_tflops,share_of_peak
0,1,640,fp32,4.785,36.527,7.055,0.678,0.084
1,1,640,fp16,4.785,73.055,8.523,0.561,0.009
4,8,640,fp32,38.282,39.252,24.909,1.537,0.190
5,8,640,fp16,38.282,78.504,12.917,2.964,0.046
8,32,640,fp32,153.126,39.568,93.883,1.631,0.201
9,32,640,fp16,153.126,79.136,48.844,3.135,0.048
10,1,320,fp32,1.276,29.986,7.501,0.170,0.021
11,1,480,fp32,2.871,34.693,7.095,0.405,0.050
12,1,640,fp32,4.785,36.527,6.978,0.686,0.085
13,1,800,fp32,7.577,37.627,7.463,1.015,0.125


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                           aten::conv2d         1.09%       3.742ms        70.08%     240.770ms     136.801us       0.000us         0.00%     106.275ms      60.384us          1760  
                                      aten::convolution         2.54%       8.725ms        68.99%     237.028ms     134.675us       0.000us         0.00%     106.275ms      60.384us          1760  
         

## 5. Accuracy vs input resolution and FP16 (VALID, paired against 640 FP32)

In [8]:
def variant_rows(paths, label_fn):
    out = []
    for p in paths:
        mm = matches(gt_valid, p); d = paired_bootstrap(mm, m_valid, headline, n_boot=500, seed=0)
        sp = np.median([sum(e["speed_ms"].values()) for e in load_json(p)["images"]])
        pc = summarise(mm, CLASS_NAMES)["per_class"]
        out.append({"variant": label_fn(p), "headline AP50": headline(mm), "Δ vs 640 FP32": d["point"], "Δ CI low": d["low"], "Δ CI high": d["high"],
                    "AP50 unsealed": pc[UNS]["ap50"], "AP50 packaging-error": pc[PKG]["ap50"],
                    "mAP50 all": summarise(mm, CLASS_NAMES)["map50"], "end-to-end ms (T4)": sp})
    return pd.DataFrame(out)
res_tab = variant_rows(sorted(R.glob("perf/valid_imgsz*.json"), key=lambda p: (int(p.stem.split("imgsz")[1].split("_")[0]), p.stem)),
                       lambda p: p.stem.replace("valid_", ""))
display(res_tab.round(3)); res_tab.to_csv(OUT / "resolution_precision.csv", index=False)
fig, ax = plt.subplots(figsize=(6.4, 3.6))
r32 = res_tab[res_tab.variant.str.endswith("fp32")]
sizes = r32.variant.str.extract(r"imgsz(\d+)")[0].astype(int)
ax.errorbar(sizes, r32["headline AP50"], yerr=[r32["Δ vs 640 FP32"] - r32["Δ CI low"], r32["Δ CI high"] - r32["Δ vs 640 FP32"]],
            marker="o", color=SERIES[0], capsize=3, label="FP32 (bars: paired 95% CI of Δ)")
f16 = res_tab[res_tab.variant.str.endswith("fp16")]
ax.scatter([640], f16["headline AP50"], color=SERIES[1], zorder=3, s=50, label="FP16 at 640")
ax.set_xlabel("input size (px); model trained at 640"); ax.set_ylabel("mean AP50, unsealed + packaging-error"); ax.legend()
ax.set_title("Accuracy vs input resolution (VALID)")
fig.tight_layout(); fig.savefig(FIG / "06_resolution.png", dpi=150); plt.show()

,variant,headline AP50,Δ vs 640 FP32,Δ CI low,Δ CI high,AP50 unsealed,AP50 packaging-error,mAP50 all,end-to-end ms (T4)
0,imgsz320_fp32,0.612,-0.135,-0.176,-0.094,0.812,0.412,0.530,9.219
1,imgsz480_fp32,0.717,-0.030,-0.079,0.023,0.829,0.604,0.590,9.350
2,imgsz640_fp16,0.744,-0.002,-0.006,-0.000,0.808,0.681,0.608,11.023
3,imgsz640_fp32,0.747,0.000,0.000,0.000,0.807,0.687,0.609,9.664
4,imgsz800_fp32,0.711,-0.035,-0.071,-0.002,0.805,0.618,0.689,10.887
5,imgsz960_fp32,0.672,-0.074,-0.118,-0.025,0.768,0.577,0.484,12.164


## 6. Robustness: controlled perturbations of VALID (paired against clean)

In [9]:
rob = variant_rows(sorted(R.glob("robustness/valid_*.json")), lambda p: p.stem.replace("valid_", ""))
rob["kind"] = rob.variant.str.split("_").str[0]; rob["level"] = rob.variant.str.split("_").str[1].astype(float)
rob = rob.sort_values(["kind", "level"])
display(rob.drop(columns=["end-to-end ms (T4)"]).round(3)); rob.to_csv(OUT / "robustness.csv", index=False)
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2), sharey=True)
clean = headline(m_valid)
labels = {"exposure": "intensity factor", "blur": "Gaussian σ (px)", "noise": "noise σ (grey levels)", "jpeg": "JPEG quality"}
for ax, kind in zip(axes, ("exposure", "blur", "noise", "jpeg")):
    s = rob[rob.kind == kind]
    ax.errorbar(s.level, s["headline AP50"], yerr=[s["Δ vs 640 FP32"] - s["Δ CI low"], s["Δ CI high"] - s["Δ vs 640 FP32"]],
                marker="o", color=SERIES[0], capsize=3)
    ax.axhline(clean, color=MUTED, ls="--", lw=1); ax.set_title(kind); ax.set_xlabel(labels[kind])
    if kind == "jpeg": ax.invert_xaxis()
axes[0].set_ylabel("mean AP50 (unsealed + pkg-error)"); axes[0].text(0.5, clean + 0.01, "clean", fontsize=8, color=MUTED)
fig.tight_layout(); fig.savefig(FIG / "06_robustness.png", dpi=150); plt.show()

,variant,headline AP50,Δ vs 640 FP32,Δ CI low,Δ CI high,AP50 unsealed,AP50 packaging-error,mAP50 all,kind,level
0,blur_1.0,0.741,-0.006,-0.032,0.015,0.810,0.671,0.595,blur,1.00
1,blur_2.0,0.725,-0.021,-0.073,0.023,0.842,0.609,0.675,blur,2.00
2,blur_4.0,0.475,-0.271,-0.342,-0.192,0.637,0.314,0.475,blur,4.00
3,exposure_0.5,0.734,-0.013,-0.034,0.008,0.813,0.655,0.677,exposure,0.50
4,exposure_0.75,0.745,-0.002,-0.022,0.020,0.820,0.671,0.613,exposure,0.75
5,exposure_1.25,0.737,-0.010,-0.031,0.008,0.802,0.673,0.599,exposure,1.25
6,exposure_1.5,0.736,-0.011,-0.031,0.011,0.812,0.659,0.617,exposure,1.50
7,jpeg_10,0.540,-0.206,-0.235,-0.171,0.796,0.284,0.507,jpeg,10.00
8,jpeg_30,0.714,-0.033,-0.058,-0.009,0.811,0.616,0.688,jpeg,30.00
9,noise_10.0,0.710,-0.037,-0.057,-0.011,0.805,0.615,0.623,noise,10.00


## 7. Save the summary used in RESULTS_LOG R6

In [10]:
summary = {"test_vs_valid": json.loads(pd.read_csv(OUT / "test_vs_valid.csv").to_json(orient="records")),
           "decision": json.loads(dec[cols].reset_index().to_json(orient="records")),
           "decision_rates_test": {"miss": float(m_rate), "false_reject": float(f_rate), "review_defective": float(qd), "review_clean": float(qc)},
           "baseline": json.loads(base.to_json(orient="records")),
           "resolution": json.loads(res_tab.to_json(orient="records")),
           "robustness": json.loads(rob.to_json(orient="records"))}
json.dump(summary, open(OUT / "summary.json", "w"), indent=1)
print("saved", OUT / "summary.json")

saved /home/claude/mma3001project/results/final/summary.json
